In [8]:
import json
import math
import random
import sys
import timm
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import matplotlib.pyplot as plt

# -------------------
# -------------------
TRAIN_JSON = Path("/zpool/vladlab/data_drive/stimulus_sets/geogaze_COCO_stim/coco_working/working_v2/instances_val_filtered2.json")
TRAIN_IMG_DIR = Path("/zpool/vladlab/data_drive/stimulus_sets/geogaze_COCO_stim/coco_working/working_v2/val_working2")  # contains 000000xxxxxx.jpg

#VAL_JSON   = Path("/path/to/annotations/instances_val2017_edited.json")  



# -------------------
# TRAINING SETTINGS
# -------------------
SEED = 123
VAL_FRACTION = 0.10   # validation is sampled from train set
BATCH_SIZE = 50
NUM_WORKERS = 4
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
IMG_SIZE = 224
PRINT_EVERY = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)


Device: cuda


In [9]:
import torchvision.transforms as T

train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

eval_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])


In [10]:
def load_coco(json_path: Path):
    with json_path.open("r") as f:
        return json.load(f)

train_coco = load_coco(TRAIN_JSON)

images = train_coco.get("images", [])
annotations = train_coco.get("annotations", [])
categories = train_coco.get("categories", [])

print("Train JSON loaded:")
print("  images:", len(images))
print("  annotations:", len(annotations))
print("  categories:", len(categories))

# Sort categories by category_id for a stable mapping
categories_sorted = sorted(categories, key=lambda c: c["id"])
cat_id_to_index = {c["id"]: i for i, c in enumerate(categories_sorted)}  # 0..79
index_to_cat = {i: c for i, c in enumerate(categories_sorted)}           # contains id/name/supercategory

# Convenience
index_to_name = {i: index_to_cat[i]["name"] for i in index_to_cat}
name_to_index = {index_to_name[i]: i for i in index_to_name}

num_classes = len(categories_sorted)
print("num_classes:", num_classes)
print("First few classes:", [(i, index_to_name[i], index_to_cat[i]["id"]) for i in range(min(10, num_classes))])


Train JSON loaded:
  images: 3552
  annotations: 7070
  categories: 80
num_classes: 80
First few classes: [(0, 'person', 1), (1, 'bicycle', 2), (2, 'car', 3), (3, 'motorcycle', 4), (4, 'airplane', 5), (5, 'bus', 6), (6, 'train', 7), (7, 'truck', 8), (8, 'boat', 9), (9, 'traffic light', 10)]


In [11]:
# Map image_id -> file_name
img_id_to_file = {im["id"]: im["file_name"] for im in images}

# image_id -> set of category indices present
img_id_to_catset = defaultdict(set)

# Optional: ignore crowd annotations
IGNORE_ISCROWD = False

missing_image_ids = 0
missing_category_ids = 0

for ann in annotations:
    if IGNORE_ISCROWD and ann.get("iscrowd", 0) == 1:
        continue

    img_id = ann["image_id"]
    cat_id = ann["category_id"]

    if img_id not in img_id_to_file:
        missing_image_ids += 1
        continue
    if cat_id not in cat_id_to_index:
        missing_category_ids += 1
        continue

    img_id_to_catset[img_id].add(cat_id_to_index[cat_id])

print("Missing image_id references in annotations:", missing_image_ids)
print("Missing category_id references in annotations:", missing_category_ids)

# Build a label vector per image_id
img_ids = [im["id"] for im in images]

labels_by_imgid = {}
for img_id in img_ids:
    y = np.zeros((num_classes,), dtype=np.float32)
    for idx in img_id_to_catset.get(img_id, set()):
        y[idx] = 1.0
    labels_by_imgid[img_id] = y

# Quick summary: how many positives per image?
pos_counts = [int(labels_by_imgid[i].sum()) for i in img_ids]
print("Images with 0 labels:", sum(c == 0 for c in pos_counts))
print("Mean labels/image:", np.mean(pos_counts))
print("Max labels/image:", np.max(pos_counts))


Missing image_id references in annotations: 0
Missing category_id references in annotations: 0
Images with 0 labels: 0
Mean labels/image: 1.563063063063063
Max labels/image: 5


In [12]:
all_ids = img_ids.copy()
random.shuffle(all_ids)

n_val = int(len(all_ids) * VAL_FRACTION)
val_ids = set(all_ids[:n_val])
train_ids = set(all_ids[n_val:])

print("Train IDs:", len(train_ids))
print("Val IDs:", len(val_ids))
print("Overlap:", len(train_ids & val_ids))


Train IDs: 3197
Val IDs: 355
Overlap: 0


In [13]:
class CocoMultiLabelDataset(Dataset):
    def __init__(self, img_dir: Path, img_id_list, img_id_to_file, labels_by_imgid, transform=None):
        self.img_dir = Path(img_dir)
        self.img_ids = list(img_id_list)
        self.img_id_to_file = img_id_to_file
        self.labels_by_imgid = labels_by_imgid
        self.transform = transform

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        file_name = self.img_id_to_file[img_id]
        path = self.img_dir / file_name

        # Load RGB image
        img = Image.open(path).convert("RGB")

        y = torch.from_numpy(self.labels_by_imgid[img_id])  # float32 multi-hot

        if self.transform is not None:
            img = self.transform(img)

        return img, y, img_id, file_name


In [14]:
train_ds = CocoMultiLabelDataset(
    img_dir=TRAIN_IMG_DIR,
    img_id_list=train_ids,
    img_id_to_file=img_id_to_file,
    labels_by_imgid=labels_by_imgid,
    transform=train_tfms
)

val_ds = CocoMultiLabelDataset(
    img_dir=TRAIN_IMG_DIR,
    img_id_list=val_ids,
    img_id_to_file=img_id_to_file,
    labels_by_imgid=labels_by_imgid,
    transform=eval_tfms
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda")
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Train batches: 64
Val batches: 8


In [15]:
import sys
from pathlib import Path
import torch
import torch.nn as nn

# Add CORnet repo to sys.path
CORN_NET_REPO = Path("/zpool/vladlab/active_drive/omaltz/git_repos/CORnet")
sys.path.insert(0, str(CORN_NET_REPO))

import cornet
print("Imported cornet from:", cornet.__file__)

def replace_last_linear(model: nn.Module, num_classes: int):
    last_name, last_linear = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            last_name, last_linear = name, module
    if last_linear is None:
        raise RuntimeError("No nn.Linear layer found in CORnet-Z.")
    parent = model
    parts = last_name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], nn.Linear(last_linear.in_features, num_classes))
    print(f"Replaced final layer: {last_name} → {num_classes} outputs")
    return model

# 1) Instantiate CORnet-Z with random weights (no downloading)
model = cornet.cornet_z(pretrained=False)

# 2) Replace classifier head for 80-way multi-label
model = replace_last_linear(model, num_classes)

# 3) Move to device
model = model.to(DEVICE)

print("CORnet-Z initialized from scratch.")


Imported cornet from: /zpool/vladlab/active_drive/omaltz/git_repos/CORnet/cornet/__init__.py
Replaced final layer: module.decoder.linear → 80 outputs
CORnet-Z initialized from scratch.
